In [1]:
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
import astropy.units as u

from spectral_cube import SpectralCube as sc
from tqdm.notebook import trange
import glob
import gc
import warnings
warnings.filterwarnings("ignore")

In [13]:
def combine_matrices(matrix1, matrix2, matrix3, matrix4):
    bottom_row = np.concatenate((matrix4, matrix3), axis=2)
    top_row    = np.concatenate((matrix2, matrix1), axis=2)
    combined_matrix = np.concatenate((top_row, bottom_row), axis=1)
    return combined_matrix

In [9]:
fits_file_list = sorted(glob.glob('./South/*.fits'))
fits_file_list

['./South/00_DEC_-13.0_-3.0__06_RA_60.0_70.0__-600kms_600kms.I.fits',
 './South/00_DEC_-13.0_-3.0__07_RA_70.0_80.0__-600kms_600kms.I.fits',
 './South/00_DEC_-13.0_-3.0__08_RA_80.0_90.0__-600kms_600kms.I.fits',
 './South/00_DEC_-13.0_-3.0__09_RA_90.0_100.0__-600kms_600kms.I.fits',
 './South/00_DEC_-13.0_-3.0__10_RA_100.0_110.0__-600kms_600kms.I.fits',
 './South/00_DEC_-13.0_-3.0__11_RA_110.0_120.0__-600kms_600kms.I.fits',
 './South/00_DEC_-13.0_-3.0__12_RA_120.0_130.0__-600kms_600kms.I.fits',
 './South/00_DEC_-13.0_-3.0__13_RA_130.0_140.0__-600kms_600kms.I.fits',
 './South/01_DEC_-3.0_7.0__06_RA_60.0_70.0__-600kms_600kms.I.fits',
 './South/01_DEC_-3.0_7.0__07_RA_70.0_80.0__-600kms_600kms.I.fits',
 './South/01_DEC_-3.0_7.0__08_RA_80.0_90.0__-600kms_600kms.I.fits',
 './South/01_DEC_-3.0_7.0__09_RA_90.0_100.0__-600kms_600kms.I.fits',
 './South/01_DEC_-3.0_7.0__10_RA_100.0_110.0__-600kms_600kms.I.fits',
 './South/01_DEC_-3.0_7.0__11_RA_110.0_120.0__-600kms_600kms.I.fits',
 './South/01_DEC_-

In [ ]:

group_info = [
    {
        "name": "./South/Combined/CRAFTS_RA60_80_DEC-13_2.fits",
        "files": [fits_file_list[0], fits_file_list[1], fits_file_list[8], fits_file_list[9]],
        "xlo": 60.0, "xhi": 80.0, "ylo": -13.0, "yhi": 2.0
    },
    {
        "name": "./South/Combined/CRAFTS_RA80_100_DEC-13_2.fits",
        "files": [fits_file_list[2], fits_file_list[3], fits_file_list[10], fits_file_list[11]],
        "xlo": 80.0, "xhi": 100.0, "ylo": -13.0, "yhi": 2.0
    },
    {
        "name": "./South/Combined/CRAFTS_RA100_120_DEC-13_2.fits",
        "files": [fits_file_list[4], fits_file_list[5], fits_file_list[12], fits_file_list[13]],
        "xlo": 100.0, "xhi": 120.0, "ylo": -13.0, "yhi": 2.0
    },
    {
        "name": "./South/Combined/CRAFTS_RA120_140_DEC-13_2.fits",
        "files": [fits_file_list[6], fits_file_list[7], fits_file_list[14], fits_file_list[15]],
        "xlo": 120.0, "xhi": 140.0, "ylo": -13.0, "yhi": 2.0
    }
]

In [4]:
for i in trange(len(group_info)):
    group = group_info[i]
    # 读取数据
    data1 = fits.getdata(group["files"][0], memmap=True)
    data2 = fits.getdata(group["files"][1], memmap=True)
    data3 = fits.getdata(group["files"][2], memmap=True)
    data4 = fits.getdata(group["files"][3], memmap=True)
    # 拼接
    combined_data = combine_matrices(data1, data2, data3, data4)
    combined_data = combined_data << u.K
    print("Combined data shape:", combined_data.shape)
    del data1, data2, data3, data4
    gc.collect()
    # 用第2张的header为基准
    hdr = fits.getheader(group["files"][1])
    # 构建cube
    cube = sc(data=combined_data, wcs=WCS(hdr))
    cube = cube[::-1]
    print("Combined cube:")
    print(cube)
    del combined_data
    gc.collect()
    # 裁剪你需要的区域
    region_cube = cube.subcube(
        xlo=group['xlo'] * u.deg,
        xhi=group['xhi'] * u.deg,
        ylo=group['ylo'] * u.deg,
        yhi=group['yhi'] * u.deg - 0.01 * u.deg
    )
    print("Region cube:")
    print(region_cube, "\n")
    region_cube.write(group['name'], overwrite=True)
    del cube, region_cube
    gc.collect()
    print(f"Saved {group['name']}")

  0%|          | 0/4 [00:00<?, ?it/s]

Combined data shape: (5962, 800, 800)
Combined cube:
SpectralCube with shape=(5962, 800, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    800  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    6.987500 deg
 n_s:   5962  type_s: VRAD      unit_s: m / s  range:  -599897.172 m / s:  599954.336 m / s
Region cube:
SpectralCube with shape=(5962, 600, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    60.012500 deg:   79.987500 deg
 n_y:    600  type_y: DEC--CAR  unit_y: deg    range:   -12.987500 deg:    1.987500 deg
 n_s:   5962  type_s: VRAD      unit_s: m / s  range:  -599897.172 m / s:  599954.336 m / s 

Saved ./South/Combined/CRAFTS_RA60_80_DEC-13_2.fits
Combined data shape: (5962, 800, 800)
Combined cube:
SpectralCube with shape=(5962, 800, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    80.012500 deg:   99.987500 deg
 n_y:    800  type_y: DEC--CAR  unit_y: deg 

In [10]:
fits_file_list = sorted(glob.glob('./North/*.fits'))
fits_file_list

['./North/03_DEC_17.0_27.0__01_RA_10.0_20.0__-600kms_600kms.I.fits',
 './North/03_DEC_17.0_27.0__02_RA_20.0_30.0__-600kms_600kms.I.fits',
 './North/03_DEC_17.0_27.0__03_RA_30.0_40.0__-600kms_600kms.I.fits',
 './North/03_DEC_17.0_27.0__04_RA_40.0_50.0__-600kms_600kms.I.fits',
 './North/03_DEC_17.0_27.0__05_RA_50.0_60.0__-600kms_600kms.I.fits',
 './North/03_DEC_17.0_27.0__06_RA_60.0_70.0__-600kms_600kms.I.fits',
 './North/03_DEC_17.0_27.0__07_RA_70.0_80.0__-600kms_600kms.I.fits',
 './North/03_DEC_17.0_27.0__08_RA_80.0_90.0__-600kms_600kms.I.fits',
 './North/04_DEC_27.0_37.0__01_RA_10.0_20.0__-600kms_600kms.I.fits',
 './North/04_DEC_27.0_37.0__02_RA_20.0_30.0__-600kms_600kms.I.fits',
 './North/04_DEC_27.0_37.0__03_RA_30.0_40.0__-600kms_600kms.I.fits',
 './North/04_DEC_27.0_37.0__04_RA_40.0_50.0__-600kms_600kms.I.fits',
 './North/04_DEC_27.0_37.0__05_RA_50.0_60.0__-600kms_600kms.I.fits',
 './North/04_DEC_27.0_37.0__06_RA_60.0_70.0__-600kms_600kms.I.fits',
 './North/04_DEC_27.0_37.0__07_RA_

In [11]:
group_info = [
    {
        "name": "./North/Combined/CRAFTS_RA10_30_DEC24_33.fits",
        "files": [fits_file_list[0], fits_file_list[1], fits_file_list[8], fits_file_list[9]],
        "xlo": 10.0, "xhi": 30.0, "ylo": 24.2, "yhi": 33.275
    },
    {
        "name": "./North/Combined/CRAFTS_RA30_50_DEC24_33.fits",
        "files": [fits_file_list[2], fits_file_list[3], fits_file_list[10], fits_file_list[11]],
        "xlo": 30.0, "xhi": 50.0, "ylo": 24.2, "yhi": 33.275
    },
    {
        "name": "./North/Combined/CRAFTS_RA50_70_DEC24_33.fits",
        "files": [fits_file_list[4], fits_file_list[5], fits_file_list[12], fits_file_list[13]],
        "xlo": 50.0, "xhi": 70.0, "ylo": 24.2, "yhi": 33.275
    },
    {
        "name": "./North/Combined/CRAFTS_RA70_90_DEC24_33.fits",
        "files": [fits_file_list[6], fits_file_list[7], fits_file_list[14], fits_file_list[15]],
        "xlo": 70.0, "xhi": 90.0, "ylo": 24.2, "yhi": 33.275
    }
]

In [14]:
for i in trange(len(group_info)):
    group = group_info[i]
    # 读取数据
    data1 = fits.getdata(group["files"][0], memmap=True)
    data2 = fits.getdata(group["files"][1], memmap=True)
    data3 = fits.getdata(group["files"][2], memmap=True)
    data4 = fits.getdata(group["files"][3], memmap=True)
    # 拼接
    combined_data = combine_matrices(data1, data2, data3, data4)
    combined_data = combined_data << u.K
    del data1, data2, data3, data4
    gc.collect()
    # 用第2张的header为基准
    hdr = fits.getheader(group["files"][1])
    # 构建cube
    cube = sc(data=combined_data, wcs=WCS(hdr))
    cube = cube[::-1]
    print("Combined cube:")
    print(cube)
    del combined_data
    gc.collect()
    # 裁剪你需要的区域
    region_cube = cube.subcube(
        xlo=group['xlo'] * u.deg,
        xhi=group['xhi'] * u.deg,
        ylo=group['ylo'] * u.deg,
        yhi=group['yhi'] * u.deg - 0.01 * u.deg
    )
    print("Region cube:")
    print(region_cube, "\n")
    region_cube.write(group['name'], overwrite=True)
    del cube, region_cube
    gc.collect()
    print(f"Saved {group['name']}")

  0%|          | 0/4 [00:00<?, ?it/s]

Combined cube:
SpectralCube with shape=(5962, 800, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    10.012500 deg:   29.987500 deg
 n_y:    800  type_y: DEC--CAR  unit_y: deg    range:    17.012500 deg:   36.987500 deg
 n_s:   5962  type_s: VRAD      unit_s: m / s  range:  -599897.172 m / s:  599954.336 m / s
Region cube:
SpectralCube with shape=(5962, 364, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    10.012500 deg:   29.987500 deg
 n_y:    364  type_y: DEC--CAR  unit_y: deg    range:    24.187500 deg:   33.262500 deg
 n_s:   5962  type_s: VRAD      unit_s: m / s  range:  -599897.172 m / s:  599954.336 m / s 

Saved ./North/Combined/CRAFTS_RA10_30_DEC24_33.fits
Combined cube:
SpectralCube with shape=(5962, 800, 800) and unit=K:
 n_x:    800  type_x: RA---CAR  unit_x: deg    range:    30.012500 deg:   49.987500 deg
 n_y:    800  type_y: DEC--CAR  unit_y: deg    range:    17.012500 deg:   36.987500 deg
 n_s:   5962  type_s: VRAD      